In [14]:

import re
import json
from pathlib import Path
import pdfplumber

ROW_PATTERN = re.compile(
    r'^(?P<item>\d+)\s+'
    r'(?P<subcode>D\d{4})\s*'
    r'(?P<paidcode>D\d{4})\s+'
    r'(?:(?P<tooth>\d{1,2}(?:\s[A-Z]{1,3})?|[A-Z]{2})\s+(?=[a-z]))?'
    r'(?P<description>.+?)\s*'
    r'(?P<dos>\d{2}/\d{2}/\d{2})\s*'
    r'\$(?P<submitted>[\d,]+\.\d{2})\s+'
    r'\$(?P<approved>[\d,]+\.\d{2})\s+'
    r'\$(?P<allowed>[\d,]+\.\d{2})\s+'
    r'\$(?P<otherinsur>[\d,]+\.\d{2})\s+'
    r'\$(?P<copay>[\d,]+\.\d{2})\s+'
    r'(?P<planpct>\d+)%\s+'
    r'\$(?P<deduct>[\d,]+\.\d{2})\s+'
    r'\$(?P<patientpay>[\d,]+\.\d{2})\s+'
    r'\$(?P<writeoff>[\d,]+\.\d{2})\s+'
    r'\$(?P<planpay>[\d,]+\.\d{2})'
    r'(?P<exc>\d{3,4}(?:[\s,]+\d{3,4})*)?\s*$',   # <-- CHANGED: allow multiple codes
    re.MULTILINE | re.DOTALL
)

TOTALS_PATTERN = re.compile(
    r'^Total:\s+\$(?P<submitted>[\d,]+\.\d{2})\s+\$(?P<approved>[\d,]+\.\d{2})\s+'
    r'\$(?P<allowed>[\d,]+\.\d{2})\s+\$(?P<otherinsur>[\d,]+\.\d{2})\s+\$(?P<copay>[\d,]+\.\d{2})\s+'
    r'\$(?P<deduct>[\d,]+\.\d{2})\s+\$(?P<patientpay>[\d,]+\.\d{2})\s+\$(?P<writeoff>[\d,]+\.\d{2})\s+'
    r'\$(?P<planpay>[\d,]+\.\d{2})\s*$',
    re.MULTILINE
)

POLICY_NOTE_PATTERN = re.compile(
    r'Processing Policy\s+(?P<code>\d{3,4})\s*-\s*(?P<note>.+?)(?=\n|\Z)'
)

LEGEND_PATTERN = re.compile(
    r'^(?P<code>\d{3,4})\s+(?P<desc>.+?)(?=\n\d{3,4}\s|\nCurrent Dental Terminology|\Z)',
    re.MULTILINE | re.DOTALL
)

def _clean(text):
    return re.sub(r'\s+', ' ', text).strip()

def _service_signature(services):
    """Signature of a patient block based on the service codes billed."""
    return tuple(
        (s["submitted_code"], s["paid_code"], s.get("tooth"), s["dos"])
        for s in services
    )

def remove_zero_duplicate_claims(patients):
    """
    If two patient blocks (same patient) have identical service-code
    signatures and one of them shows $0.00 for BOTH total_allowed_amount
    and total_plan_pay, drop that zero block entirely.
    """
    from collections import defaultdict

    groups = defaultdict(list)
    for idx, p in enumerate(patients):
        sig = (p["patient_name"], _service_signature(p["services"]))
        groups[sig].append(idx)

    remove_idxs = set()
    for sig, idxs in groups.items():
        if len(idxs) < 2:
            continue
        for idx in idxs:
            totals = patients[idx].get("totals") or {}
            allowed = totals.get("total_allowed_amount", "$0.00")
            planpay = totals.get("total_plan_pay", "$0.00")
            if allowed == "$0.00" and planpay == "$0.00":
                remove_idxs.add(idx)

    return [p for i, p in enumerate(patients) if i not in remove_idxs]


def parse_amount(value):
    """Convert a currency string like '$1,234.50' to a float. Empty/None -> 0.0"""
    if value is None:
        return 0.0
    if isinstance(value, (int, float)):
        return float(value)
    cleaned = re.sub(r'[^0-9.\-]', '', str(value))
    if cleaned in ("", "-", "."):
        return 0.0
    try:
        return float(cleaned)
    except ValueError:
        return 0.0

# service-field -> totals-field
SERVICE_TO_TOTAL_FIELD = {
    "submitted_amount":  "total_submitted_amount",
    "approved_amount":   "total_approved_amount",
    "allowed_amount":    "total_allowed_amount",
    "other_insurance":   "total_other_insurance",
    "copay_amount":      "total_copay_amount",
    "deductible_amount": "total_deductible_amount",
    "patient_pay":       "total_patient_pay",
    "writeoff_amount":   "total_writeoff_amount",
    "plan_pay":          "total_plan_pay",
}

def compute_totals_from_services(services):
    return {
        total_field: round(sum(parse_amount(s.get(svc_field, "")) for s in services), 2)
        for svc_field, total_field in SERVICE_TO_TOTAL_FIELD.items()
    }

def validate_patient_totals(patient, patient_name=None):
    services = patient.get("services", [])
    totals = patient.get("totals") or {}
    patient_name = patient_name or patient.get("patient_name") or "UNKNOWN"

    field_names = list(SERVICE_TO_TOTAL_FIELD.values())
    total_fields = len(field_names) + 1  # +1 for row-count check

    if not services:
        field_errors = [{"field": f, "computed": 0.0, "extracted": None} for f in field_names]
        print(f"\n🔍 Validation for [{patient_name}]")
        print("-" * 80)
        print("⚠️  No services found — skipping validation")
        print("-" * 80)
        return False, "No services found", [{"error": "empty services"}] + field_errors, total_fields

    computed_totals = compute_totals_from_services(services)

    result_validation = ""
    errors = []
    has_error = False

    print(f"\n🔍 Validation for [{patient_name}]")
    print("-" * 80)

    for field, computed_value in computed_totals.items():
        extracted_value = round(parse_amount(totals.get(field, "")), 2)
        diff = round(computed_value - extracted_value, 2)
        match = abs(diff) <= 0.01

        if match:
            icon, status = "✅", "MATCH"
        else:
            icon, status = "❌", "MISMATCH"
            has_error = True
            errors.append({
                "type": "field_mismatch",
                "field": field,
                "computed": computed_value,
                "extracted": extracted_value,
                "difference": diff
            })

        line = f"{icon} {field:25s} computed={computed_value:<10} | extracted={extracted_value:<10} {status}"
        print(line)
        result_validation += "\n" + line

    # Row count: items are numbered sequentially, so the last item number
    # should equal how many services we actually parsed.
    try:
        expected_row_count = int(services[-1]["item"])
    except (ValueError, TypeError, KeyError, IndexError):
        expected_row_count = len(services)

    extracted_row_count = len(services)

    if expected_row_count == extracted_row_count:
        icon, status = "✅", "MATCH"
    else:
        icon, status = "❌", "MISMATCH"
        has_error = True
        errors.append({
            "type": "row_count_mismatch",
            "expected_rows": expected_row_count,
            "extracted_rows": extracted_row_count
        })

    line = f"{icon} {'total_record_rows':25s} computed={expected_row_count:<10} | extracted={extracted_row_count:<10} {status}"
    print(line)
    result_validation += "\n" + line

    print("-" * 80)

    if has_error:
        print(f"❌ [{patient_name}] Validation FAILED\n")
        return False, result_validation, errors, total_fields
    else:
        print(f"✅ [{patient_name}] Validation PASSED\n")
        return True, result_validation, [], total_fields


def extract_dentaquest_pdf(pdf_path):
    pdf_path = Path(pdf_path)
    with pdfplumber.open(pdf_path) as pdf:
        patient_texts = []
        legend_texts = []
        for page in pdf.pages:
            if page.width <= page.height:
                continue
            text = page.extract_text() or ""

            if "Patient Name:" in text:
                patient_texts.append(text)

            # CHANGED: no longer "elif" — a page can have BOTH patient blocks
            # AND a "Processing Policies Description" legend at the bottom.
            if "Processing Policies Description" in text:
                idx = text.index("Processing Policies Description")
                legend_texts.append(text[idx:])       # only the legend section
            elif "Claim Detail" in text:
                legend_texts.append(text)             # standalone legend-only pages

    legend = {}
    for legend_text in legend_texts:
        for m in LEGEND_PATTERN.finditer(legend_text):
            legend[m.group("code")] = _clean(m.group("desc"))

    full_text = "\n".join(patient_texts)
    provider_m = re.search(r'Provider Name:\s*(.+?)\s+Office Reference', full_text)
    provider_name = provider_m.group(1).strip() if provider_m else None
    raw_blocks = [b for b in re.split(r'(?=Patient Name:)', full_text) if b.strip().startswith("Patient Name:")]
    parsed = [_parse_block(b) for b in raw_blocks]

    merged = []
    for p in parsed:
        if (merged and merged[-1]["patient_name"] == p["patient_name"]
                and merged[-1]["claim_number"] == p["claim_number"]
                and p["claim_number"] is not None):
            merged[-1]["services"].extend(p["services"])
            if p["totals"]:
                merged[-1]["totals"] = p["totals"]
        else:
            merged.append(p)

    merged = remove_zero_duplicate_claims(merged)

    for patient in merged:
        for svc in patient["services"]:
            code = svc["exc_code"]
            if code:
                codes = [c.strip() for c in code.split(",") if c.strip()]
                svc["denial_status"] = "Denied"
                reasons = []
                for c in codes:
                    reason_text = legend.get(c)
                    reasons.append(f"{c} - {reason_text}" if reason_text else c)
                svc["denial_reason"] = "; ".join(reasons)
            else:
                svc["denial_status"] = "Not Denied"
                svc["denial_reason"] = None

    for patient in merged:
        is_valid, report, errs, total_fields = validate_patient_totals(patient)
        patient["validation"] = {
            "passed": is_valid,
            "errors": errs,
            # "total_fields_checked": total_fields
        }

    eob_id = pdf_path.stem

    return {
        "eob_id": eob_id,
        "file_name": pdf_path.name,
        "provider_name": provider_name,
        "confidence_score": 100.0,
        "patients": merged
    }

def _parse_block(block):
    name_m = re.search(r'Patient Name:\s*(.+?)\s+Provider Name:', block)
    prov_m = re.search(r'Provider Name:\s*(.+?)\s+Office Reference', block)
    claim_m = re.search(r'Claim #:\s*(\d+)', block)
    dob_m = re.search(r'DOB:\s*(\d{2}/\d{2}/\d{4})', block)

    policy_notes = [(m.group("code"), _clean(m.group("note"))) for m in POLICY_NOTE_PATTERN.finditer(block)]
    note_iter = iter(policy_notes)
    next_note = next(note_iter, None)

    services = []
    for m in ROW_PATTERN.finditer(block):
        exc_raw = m.group("exc")
        exc_codes = [c.strip() for c in re.split(r'[,\s]+', exc_raw.strip())] if exc_raw else []
        exc = ", ".join(exc_codes) if exc_codes else None   # normalized display string

        note_detail = None
        if exc_codes and next_note and next_note[0] in exc_codes:   # CHANGED: membership check
            note_detail = next_note[1]
            next_note = next(note_iter, None)

        tail = block[m.end():]
        tail_m = re.match(
            r'(?P<extra>(?:(?!\n\d+\s+D\d{4}|\nTotal:|\nProcessing Policy|\Z).)*)',
            tail, re.DOTALL
        )
        extra = _clean(tail_m.group("extra")) if tail_m else ""
        description = _clean(m.group("description"))
        if extra:
            description = f"{description} {extra}".strip()

        services.append({
            "item": m.group("item"),
            "submitted_code": m.group("subcode"),
            "paid_code": m.group("paidcode"),
            "tooth": m.group("tooth"),
            "description": description,
            "dos": m.group("dos"),
            "submitted_amount": f'${m.group("submitted")}',
            "approved_amount": f'${m.group("approved")}',
            "allowed_amount": f'${m.group("allowed")}',
            "other_insurance": f'${m.group("otherinsur")}',
            "copay_amount": f'${m.group("copay")}',
            "plan_pct": f'{m.group("planpct")}%',
            "deductible_amount": f'${m.group("deduct")}',
            "patient_pay": f'${m.group("patientpay")}',
            "writeoff_amount": f'${m.group("writeoff")}',
            "plan_pay": f'${m.group("planpay")}',
            "exc_code": exc   # normalized "2040, 2052" or "2052" or None
        })

    totals_m = TOTALS_PATTERN.search(block)
    if totals_m:
        totals = {
            "total_submitted_amount": f'${totals_m.group("submitted")}',
            "total_approved_amount": f'${totals_m.group("approved")}',
            "total_allowed_amount": f'${totals_m.group("allowed")}',
            "total_other_insurance": f'${totals_m.group("otherinsur")}',
            "total_copay_amount": f'${totals_m.group("copay")}',
            "total_deductible_amount": f'${totals_m.group("deduct")}',
            "total_patient_pay": f'${totals_m.group("patientpay")}',
            "total_writeoff_amount": f'${totals_m.group("writeoff")}',
            "total_plan_pay": f'${totals_m.group("planpay")}'
        }
    else:
        totals = None

    return {
        "patient_name": name_m.group(1).strip() if name_m else None,
        # "provider_name": prov_m.group(1).strip() if prov_m else None,
        "claim_number": claim_m.group(1) if claim_m else None,
        "dob": dob_m.group(1) if dob_m else None,
        "services": services,
        "totals": totals
    }

def flatten_for_table(pdf_result):
    rows = []
    for patient in pdf_result["patients"]:
        base = {
            "eob_id": pdf_result["eob_id"],
            "file_name": pdf_result["file_name"],
            "provider_name": pdf_result["provider_name"],
            "confidence_score": pdf_result["confidence_score"],
            "patient_name": patient["patient_name"],
            "claim_number": patient["claim_number"],
            "dob": patient["dob"],
            **(patient["totals"] or {})
        }
        if patient["services"]:
            for svc in patient["services"]:
                row = dict(base)
                row.update(svc)
                rows.append(row)
        else:
            rows.append(base)
    return rows

def run_batch(input_dir, output_path):
    input_dir = Path(input_dir)
    output_path = Path(output_path)
    pdf_files = sorted(input_dir.glob("*.pdf"))
    if not pdf_files:
        print(f"No PDF files found in {input_dir}")
        return

    all_rows, all_results, errors = [], [], []
    for pdf_path in pdf_files:
        try:
            result = extract_dentaquest_pdf(pdf_path)
            all_results.append(result)
            all_rows.extend(flatten_for_table(result))
            n_patients = len(result["patients"])
            n_services = sum(len(p["services"]) for p in result["patients"])
            n_denied = sum(1 for p in result["patients"] for s in p["services"] if s["denial_status"] == "Denied")
            flag = "" if n_patients else "  <-- NO PATIENT BLOCKS FOUND, CHECK THIS FILE"
            print(f"OK   {pdf_path.name}: {n_patients} patient(s), {n_services} service(s), {n_denied} with processing-policy code{flag}")
        except Exception as e:
            errors.append((pdf_path.name, str(e)))
            print(f"FAIL {pdf_path.name}: {e}")

    columns = [
        "file_name", "patient_name", "provider_name", "claim_number", "dob",
        "item", "submitted_code", "paid_code", "tooth", "description", "dos",
        "submitted_amount", "approved_amount", "allowed_amount",
        "other_insurance", "copay_amount", "plan_pct", "deductible_amount",
        "patient_pay", "writeoff_amount", "plan_pay",
        "exc_code", "denial_status", "denial_reason",
        "total_submitted_amount", "total_approved_amount", "total_allowed_amount",
        "total_other_insurance", "total_copay_amount", "total_deductible_amount",
        "total_patient_pay", "total_writeoff_amount", "total_plan_pay"
    ]

    if output_path.suffix.lower() == ".xlsx":
        import openpyxl
        from openpyxl.utils import get_column_letter
        wb = openpyxl.Workbook()
        ws = wb.active
        ws.title = "DentaQuest"
        ws.append(columns)
        for row in all_rows:
            ws.append([row.get(c, "") for c in columns])
        for i, col in enumerate(columns, start=1):
            max_len = max([len(col)] + [len(str(r.get(col, ""))) for r in all_rows])
            ws.column_dimensions[get_column_letter(i)].width = min(max_len + 2, 45)
        wb.save(output_path)
    else:
        import csv
        with open(output_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=columns)
            writer.writeheader()
            for row in all_rows:
                writer.writerow(row)

    json_path = output_path.with_suffix(".json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)

    print(f"\nDone. {len(pdf_files)} PDF(s) processed, {len(errors)} failed.")
    print(f"Table written to: {output_path}")
    print(f"Raw JSON written to: {json_path}")
    if errors:
        print("\nFiles that failed:")
        for name, err in errors:
            print(f"  - {name}: {err}")

def debug_unmatched_rows(pdf_path):
    """
    Finds lines that LOOK like a service row (start with an item number
    followed by two D-codes) but weren't captured by ROW_PATTERN.
    Prints the raw text around them so we can see exactly what
    pdfplumber extracted.
    """
    import pdfplumber

    LOOSE_ITEM_LINE = re.compile(r'^\d+\s+D\d{4}\s*D\d{4}', re.MULTILINE)

    pdf_path = Path(pdf_path)
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            if page.width <= page.height:
                continue
            text = page.extract_text() or ""
            if "Patient Name:" not in text:
                continue

            matched_starts = {m.start() for m in ROW_PATTERN.finditer(text)}
            loose_starts = [m.start() for m in LOOSE_ITEM_LINE.finditer(text)]

            for start in loose_starts:
                if start not in matched_starts:
                    # print the raw text of this row and ~300 chars after it
                    snippet = text[start:start + 400]
                    print(f"\n{'='*70}")
                    print(f"PAGE {page_num} — UNMATCHED ROW (repr, to show exact whitespace):")
                    print(repr(snippet))
                    print(f"{'='*70}")

# Usage:
# debug_unmatched_rows("your_file.pdf")



def run_pipeline(pdf_path, output_root="./Test_4"):
    pdf_path = Path(pdf_path)
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    result = extract_dentaquest_pdf(pdf_path)
    folder_name = pdf_path.stem
    output_folder = Path(output_root) / folder_name
    output_folder.mkdir(parents=True, exist_ok=True)

    json_path = output_folder / f"{folder_name}.json"
    excel_path = output_folder / f"{folder_name}.xlsx"

    # JSON
    json_result = {
        "eob_id": result["eob_id"],
        "file_name": result["file_name"],
        "provider_name": result["provider_name"],
        "confidence_score": result["confidence_score"],
        "patients": []
    }

    for patient in result["patients"]:
        json_result["patients"].append({
            "patient_name": patient["patient_name"],
            "claim_number": patient["claim_number"],
            "dob": patient["dob"],
            "services": patient["services"],
            "totals": patient["totals"],
            "validation": patient["validation"]
        })

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump([json_result], f, indent=2, ensure_ascii=False)

    # Excel
    import openpyxl
    from openpyxl.utils import get_column_letter

    rows = flatten_for_table(result)

    columns = [
        "eob_id", "file_name", "provider_name", "confidence_score",
        "patient_name", "claim_number", "dob",
        "item", "submitted_code", "paid_code", "tooth", "description", "dos",
        "submitted_amount", "approved_amount", "allowed_amount",
        "other_insurance", "copay_amount", "plan_pct", "deductible_amount",
        "patient_pay", "writeoff_amount", "plan_pay",
        "exc_code", "denial_status", "denial_reason",
        "total_submitted_amount", "total_approved_amount", "total_allowed_amount",
        "total_other_insurance", "total_copay_amount", "total_deductible_amount",
        "total_patient_pay", "total_writeoff_amount", "total_plan_pay"
    ]

    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "DentaQuest"
    ws.append(columns)

    for row in rows:
        ws.append([row.get(c, "") for c in columns])

    for i, col in enumerate(columns, start=1):
        max_len = max([len(col)] + [len(str(r.get(col, ""))) for r in rows])
        ws.column_dimensions[get_column_letter(i)].width = min(max_len + 2, 45)

    # wb.save(excel_path)

    n_patients = len(result["patients"])
    n_services = sum(len(p["services"]) for p in result["patients"])
    n_denied = sum(
        1 for p in result["patients"]
        for s in p["services"]
        if s["denial_status"] == "Denied"
    )

    print("=" * 70)
    print(f"PDF      : {pdf_path.name}")
    print(f"Patients : {n_patients}")
    print(f"Services : {n_services}")
    print(f"Denied   : {n_denied}")
    print(f"JSON     : {json_path}")
    # print(f"Excel    : {excel_path}")
    print("=" * 70)

    return result, json_path, #excel_path

In [15]:
pdf_path = r"/home/cipl/users/Jeeva/Phase_2_pdf/ALL_NEW/Dentaquest/Amerihealth/37036040.pdf"
result, json_path = run_single(pdf_path, output_root="./error")


🔍 Validation for [NIVAR, VIOLETA]
--------------------------------------------------------------------------------
❌ total_submitted_amount    computed=200.0      | extracted=325.0      MISMATCH
✅ total_approved_amount     computed=57.0       | extracted=57.0       MATCH
✅ total_allowed_amount      computed=57.0       | extracted=57.0       MATCH
✅ total_other_insurance     computed=0.0        | extracted=0.0        MATCH
✅ total_copay_amount        computed=0.0        | extracted=0.0        MATCH
✅ total_deductible_amount   computed=0.0        | extracted=0.0        MATCH
✅ total_patient_pay         computed=0.0        | extracted=0.0        MATCH
❌ total_writeoff_amount     computed=143.0      | extracted=268.0      MISMATCH
✅ total_plan_pay            computed=57.0       | extracted=57.0       MATCH
✅ total_record_rows         computed=2          | extracted=2          MATCH
--------------------------------------------------------------------------------
❌ [NIVAR, VIOLETA] Validati

In [16]:
pdf_path = r"/home/cipl/users/Jeeva/Phase_2_pdf/ALL_NEW/Dentaquest/Amerihealth/37006152.pdf"
result, json_path = run_single(pdf_path, output_root="./error")


🔍 Validation for [LABOY, SOCORRO]
--------------------------------------------------------------------------------
✅ total_submitted_amount    computed=500.0      | extracted=500.0      MATCH
✅ total_approved_amount     computed=190.0      | extracted=190.0      MATCH
✅ total_allowed_amount      computed=190.0      | extracted=190.0      MATCH
✅ total_other_insurance     computed=0.0        | extracted=0.0        MATCH
✅ total_copay_amount        computed=0.0        | extracted=0.0        MATCH
✅ total_deductible_amount   computed=0.0        | extracted=0.0        MATCH
✅ total_patient_pay         computed=0.0        | extracted=0.0        MATCH
✅ total_writeoff_amount     computed=310.0      | extracted=310.0      MATCH
✅ total_plan_pay            computed=190.0      | extracted=190.0      MATCH
✅ total_record_rows         computed=4          | extracted=4          MATCH
--------------------------------------------------------------------------------
✅ [LABOY, SOCORRO] Validation PAS

In [11]:
from pathlib import Path

input_folder = Path("/home/cipl/users/Jeeva/Phase_2_pdf/ALL_NEW/Dentaquest/Amerihealth")
output_root = Path("./Test_5_42")
output_root.mkdir(parents=True, exist_ok=True)

pdf_files = sorted(input_folder.glob("*.pdf"))
total_pdf = len(pdf_files)
processed = 0
failed = 0

print(f"Total PDF       : {total_pdf}")
print(f"Processed       : {processed}")
print(f"Remaining       : {total_pdf - processed}")
print("=" * 60)

for i, pdf_path in enumerate(pdf_files, 1):
    try:
        result, json_path = run_pipeline(
            pdf_path,
            output_root=str(output_root)
        )
        processed += 1
        print(f"[{i}/{total_pdf}] Processed : {pdf_path.name}")
    except Exception as e:
        failed += 1
        print(f"[{i}/{total_pdf}] FAILED    : {pdf_path.name} -> {e}")

    remaining = total_pdf - processed - failed
    print(f"Progress -> Total: {total_pdf} | Processed: {processed} | Failed: {failed} | Remaining: {remaining}")

print("=" * 60)
print(f"Total PDF       : {total_pdf}")
print(f"Processed       : {processed}")
print(f"Failed          : {failed}")
print(f"Remaining       : {total_pdf - processed - failed}")
print(f"Output folder   : {output_root}")

Total PDF       : 42
Processed       : 0
Remaining       : 42

🔍 Validation for [AYALA, RICARDO]
--------------------------------------------------------------------------------
✅ total_submitted_amount    computed=250.0      | extracted=250.0      MATCH
✅ total_approved_amount     computed=82.39      | extracted=82.39      MATCH
✅ total_allowed_amount      computed=82.39      | extracted=82.39      MATCH
✅ total_other_insurance     computed=0.0        | extracted=0.0        MATCH
✅ total_copay_amount        computed=0.0        | extracted=0.0        MATCH
✅ total_deductible_amount   computed=0.0        | extracted=0.0        MATCH
✅ total_patient_pay         computed=0.0        | extracted=0.0        MATCH
✅ total_writeoff_amount     computed=167.61     | extracted=167.61     MATCH
✅ total_plan_pay            computed=82.39      | extracted=82.39      MATCH
✅ total_record_rows         computed=4          | extracted=4          MATCH
----------------------------------------------------